# 3. Ewaluacja LLM'a w Langfuse

Sprawdzam, jak dobrze LLM wyciąga z przedstawienia się użytkownika płeć, wiek i czas na 5 km.
Robię to tak jak w lekcji `33__monitorowanie_modeli__langfuse.ipynb`:
1. tworzę dataset w Langfuse z przykładowymi wiadomościami i oczekiwanym wynikiem,
2. uruchamiam eksperymenty dla kilku modeli,
3. każdy wynik oceniam funkcją `output_evaluator` i zapisuję jako score.

Prompt jest skopiowany z `app.py` - jeśli zmienię go w aplikacji, muszę go zmienić także tutaj.

In [1]:
#!pip install --quiet openai langfuse python-dotenv pandas

In [2]:
import os
import json
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv

from langfuse import Langfuse
from langfuse.decorators import observe
from langfuse.openai import OpenAI as LangfuseOpenAI

load_dotenv()

# Inicjacja Langfuse i sprawdzenie kluczy - jak w lekcji 33
langfuse = Langfuse()
langfuse.auth_check()

True

In [3]:
# Klient z obserwacją (wariant z dekoratorem z lekcji 33)
llm_client = LangfuseOpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

In [4]:
# Prompt skopiowany z app.py
EXTRACTION_PROMPT = """
You are awesome helper in parsing the data about a runner.
You will be provided with the user self-introduction in polish that contains the following data:

<plec> - the gender of the user. Return "K" for a woman and "M" for a man.
Polish grammar can help you (e.g. "biegałam" - woman, "biegałem" - man).
<wiek> - the age of the user in years. If the user gives the year of birth instead,
calculate the age using the current year provided at the end of this message.
<czas_5km> - the time in which the user runs 5 kilometers. Convert it to the format "HH:MM:SS",
e.g. "25 minut" -> "00:25:00", "24:30" -> "00:24:30", "pół godziny" -> "00:30:00".

Return the value as a dictionary with the following keys:
- plec - as a string "K" or "M" or null if not provided
- wiek - as an integer or null if not provided
- czas_5km - as a string in format "HH:MM:SS" or null if not provided

Do not guess. If the information is not provided, return null for that key.
Return a valid dictionary, nothing else.
I will parse the result with json.loads() function in python so please make sure the result is valid.

Here is an example of the user content:
```
Cześć, mam na imię Ania, mam 32 lata i 5 km biegam w 27 minut.
```

In this case the dictionary should look like this:
{
"plec": "K",
"wiek": 32,
"czas_5km": "00:27:00"
}
"""

In [5]:
@observe()
def get_data_from_intro_observed(intro, model="gpt-4o-mini"):
    messages = [
        {
            "role": "system",
            "content": EXTRACTION_PROMPT + f"\nThe current year is {datetime.now().year}.",
        },
        {
            "role": "user",
            "content": f"```{intro}```",
        },
    ]
    chat_completion = llm_client.chat.completions.create(
        response_format={"type": "json_object"},
        messages=messages,
        model=model,
        name="get_data_from_intro_observed",
    )
    resp = chat_completion.choices[0].message.content
    try:
        output = json.loads(resp)
    except:
        output = {"error": resp}
    return output

In [6]:
# Szybki test na jednym przykładzie
get_data_from_intro_observed("Hej, jestem Marek, mam 41 lat, a piątkę biegam w 23:45.")

{'plec': 'M', 'wiek': 41, 'czas_5km': '00:23:45'}

## Tworzę dataset

In [7]:
dataset_name = "polmaraton-przedstawienia"

intros = [
    "Cześć, mam na imię Kasia, mam 34 lata i 5 km przebiegam w 28 minut.",
    "Hej, jestem Tomek, 45 lat. Piątkę robię w 22:30.",
    "Dzień dobry, urodziłam się w 1990 roku, a 5 km biegam w pół godziny.",
    "Siema, mam 27 lat i biegam 5 km w 19 minut i 40 sekund.",
    "Cześć, jestem Ola i mam 52 lata.",
    "Hej, 5 km biegam w 31 minut, jestem mężczyzną.",
    "Jestem Piotr, biegam od niedawna.",
    "Dzień dobry, nazywam się Anna Nowak, mam 61 lat, a dystans 5 km pokonuję w 35:10.",
]

expected_outputs = [
    {"plec": "K", "wiek": 34, "czas_5km": "00:28:00"},
    {"plec": "M", "wiek": 45, "czas_5km": "00:22:30"},
    {"plec": "K", "wiek": datetime.now().year - 1990, "czas_5km": "00:30:00"},
    {"plec": None, "wiek": 27, "czas_5km": "00:19:40"},
    {"plec": "K", "wiek": 52, "czas_5km": None},
    {"plec": "M", "wiek": None, "czas_5km": "00:31:00"},
    {"plec": "M", "wiek": None, "czas_5km": None},
    {"plec": "K", "wiek": 61, "czas_5km": "00:35:10"},
]

In [8]:
# Dodatek: dataset tworzę tylko raz - przy ponownym uruchomieniu notebooka nie chcę dublować przykładów.
# Przy pierwszym uruchomieniu langfuse wypisuje błąd, bo datasetu jeszcze nie ma - to normalne.
try:
    dataset = langfuse.get_dataset(dataset_name)
    print(f"Dataset {dataset_name} już istnieje, ma {len(dataset.items)} przykładów")
except Exception:
    langfuse.create_dataset(name=dataset_name)
    for intro, output in zip(intros, expected_outputs):
        langfuse.create_dataset_item(
            dataset_name=dataset_name,
            input=intro,
            expected_output=output,
        )
    langfuse.flush()
    print(f"Utworzyłem dataset {dataset_name}")

Dataset polmaraton-przedstawienia już istnieje, ma 8 przykładów


## Uruchamiam eksperymenty na datasecie

In [9]:
# Funkcja oceniająca z lekcji 33, dostosowana do moich pól
def output_evaluator(expected_output, output):
    score = 0.0
    fields = ["plec", "wiek", "czas_5km"]
    for field in fields:
        if expected_output.get(field) == output.get(field):
            score += 1.0 / len(fields)
    return score

In [10]:
dataset = langfuse.get_dataset(dataset_name)
models = ["gpt-4o-mini", "gpt-4o"]
results = []

for model in models:
    for item in dataset.items:

        with item.observe(
            run_name=f"ekstrakcja-danych-biegacza-{model}",
            run_description=f"Ekstrakcja płci, wieku i czasu na 5 km - model {model}",
            run_metadata={"model": model},
        ) as trace_id:

            output = get_data_from_intro_observed(intro=item.input, model=model)
            score = output_evaluator(item.expected_output, output)

            langfuse.score(
                trace_id=trace_id,
                name="extraction-score",
                value=score,
                comment="Jak dobrze llm radzi sobie z wyciągnięciem danych biegacza",
            )
            results.append({"model": model, "input": item.input, "score": score, **output})

langfuse.flush()

In [11]:
# Dodatek: podsumowanie lokalnie - te same wyniki widzę w Langfuse w zakładce Datasets
results_df = pd.DataFrame(results)
results_df.groupby("model")["score"].mean()

model
gpt-4o         1.0
gpt-4o-mini    1.0
Name: score, dtype: float64

In [12]:
results_df

,model,input,score,plec,wiek,czas_5km
0,gpt-4o-mini,"Dzień dobry, nazywam się Anna Nowak, mam 61 la...",1.0,K,61.0,00:35:10
1,gpt-4o-mini,"Jestem Piotr, biegam od niedawna.",1.0,M,NaN,None
2,gpt-4o-mini,"Hej, 5 km biegam w 31 minut, jestem mężczyzną.",1.0,M,NaN,00:31:00
3,gpt-4o-mini,"Cześć, jestem Ola i mam 52 lata.",1.0,K,52.0,None
4,gpt-4o-mini,"Siema, mam 27 lat i biegam 5 km w 19 minut i 4...",1.0,None,27.0,00:19:40
5,gpt-4o-mini,"Dzień dobry, urodziłam się w 1990 roku, a 5 km...",1.0,K,36.0,00:30:00
6,gpt-4o-mini,"Hej, jestem Tomek, 45 lat. Piątkę robię w 22:30.",1.0,M,45.0,00:22:30
7,gpt-4o-mini,"Cześć, mam na imię Kasia, mam 34 lata i 5 km p...",1.0,K,34.0,00:28:00
8,gpt-4o,"Dzień dobry, nazywam się Anna Nowak, mam 61 la...",1.0,K,61.0,00:35:10
9,gpt-4o,"Jestem Piotr, biegam od niedawna.",1.0,M,NaN,None
